In [32]:
# Mini Proje: Bank Marketing Churn Prediction — Uçtan Uca
"""
Bu notebook’ta:
1) Veri yükleme & EDA  
2) Preprocessing Pipeline  
3) Model eğitimi & karşılaştırma (RF, GB, XGBoost)  
4) SHAP analizi (summary_plot, force_plot ile 3 müşteri profili, dependence_plot duration vs age)  
5) Çıktıların kaydedilmesi  
"""

'\nBu notebook’ta:\n1) Veri yükleme & EDA  \n2) Preprocessing Pipeline  \n3) Model eğitimi & karşılaştırma (RF, GB, XGBoost)  \n4) SHAP analizi (summary_plot, force_plot ile 3 müşteri profili, dependence_plot duration vs age)  \n5) Çıktıların kaydedilmesi  \n'

In [33]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib
import shap

# XGBoost (kurulu değilse: pip install xgboost)
from xgboost import XGBClassifier

In [35]:

df = pd.read_csv("bank-full.csv", sep=";")

print("Shape:", df.shape)
df.head()

Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [36]:
# Genel yapı
df.info()

# Hedef dağılımı (class imbalance)
print("\nTarget dağılımı:")
print(df["y"].value_counts())
print("\nOran:")
print(df["y"].value_counts(normalize=True))

# Eksik değer (NaN) var mı?
print("\nNaN sayıları:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB

Target dağılımı:
y
no     39922
yes     5289
Name: count, dtype: int64

Oran:
y
no     0.883015
yes    0.116985
Name: proportion, dt

In [37]:
# Sayısal ve kategorik kolonları ayır
num_cols = df.select_dtypes(include=["int64"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Target'ı kategorikten çıkar
cat_cols.remove("y")

print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

Numeric cols: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical cols: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\User\AppData\Local\Temp\ipykernel_3888\1297791013.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns.tolist()


In [38]:
X = df.drop("y", axis=1)
y = df["y"].map({"no": 0, "yes": 1})  # binary target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (36168, 16) Test: (9043, 16)


In [39]:
class OutlierClipper(BaseEstimator, TransformerMixin):
    """
    IQR tabanlı outlier clipping.
    Q1 - 1.5*IQR altını ve Q3 + 1.5*IQR üstünü kırpar.
    """
    def __init__(self, factor=1.5):
        self.factor = factor
        
    def fit(self, X, y=None):
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        return self
    
    def transform(self, X):
        lower = self.q1_ - self.factor * self.iqr_
        upper = self.q3_ + self.factor * self.iqr_
        return np.clip(X, lower, upper)

In [40]:
# Numeric pipeline: median impute + outlier clip + robust scale
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", OutlierClipper(factor=1.5)),
    ("scaler", RobustScaler())
])

# Categorical pipeline: most_frequent impute + one-hot
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Birleştirme
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [41]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [42]:
# 1) RandomForest (class imbalance için class_weight='balanced')
rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# 2) GradientBoosting (sklearn’in klasik GB’si)
gb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

# 3) XGBoost (class imbalance için scale_pos_weight)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=600,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42
    ))
])

# ROC-AUC ile CV skorları
rf_score  = cross_val_score(rf_pipeline,  X_train, y_train, cv=cv, scoring="roc_auc").mean()
gb_score  = cross_val_score(gb_pipeline,  X_train, y_train, cv=cv, scoring="roc_auc").mean()
xgb_score = cross_val_score(xgb_pipeline, X_train, y_train, cv=cv, scoring="roc_auc").mean()

results_df = pd.DataFrame({
    "Model": ["RandomForest", "GradientBoosting", "XGBoost"],
    "ROC_AUC_CV": [rf_score, gb_score, xgb_score]
}).sort_values("ROC_AUC_CV", ascending=False)

results_df

,Model,ROC_AUC_CV
2,XGBoost,0.934792
0,RandomForest,0.928869
1,GradientBoosting,0.923898


In [43]:
 # En iyi model hangisi ise onu seçiyoruz (genelde XGB çıkar)
best_name = results_df.iloc[0]["Model"]
print("Best model:", best_name)

if best_name == "RandomForest":
    best_pipeline = rf_pipeline
elif best_name == "GradientBoosting":
    best_pipeline = gb_pipeline
else:
    best_pipeline = xgb_pipeline

# Train üzerinde eğit
best_pipeline.fit(X_train, y_train)
print("Best pipeline trained.")

Best model: XGBoost
Best pipeline trained.


In [44]:
# En iyi pipeline
joblib.dump(best_pipeline, "best_pipeline.pkl")

# Model karşılaştırma tablosu
results_df.to_csv("model_comparison.csv", index=False)

print("Kaydedildi: best_pipeline.pkl")
print("Kaydedildi: model_comparison.csv")

Kaydedildi: best_pipeline.pkl
Kaydedildi: model_comparison.csv


In [46]:
# Pipeline içinden preprocessor ve model
pre = best_pipeline.named_steps["preprocess"]
model = best_pipeline.named_steps["model"]

# --- Feature names manuel üretimi (clipper get_feature_names_out desteklemediği için) ---
# Numeric taraf: bire bir kolon isimleri (num__ prefix ile)
num_feature_names = [f"num__{c}" for c in num_cols]

# Categorical taraf: OneHotEncoder'dan gerçek dummy kolon isimleri
ohe = pre.named_transformers_["cat"].named_steps["encoder"]
cat_feature_names = [f"cat__{c}" for c in ohe.get_feature_names_out(cat_cols)]

feature_names = np.array(num_feature_names + cat_feature_names, dtype=object)

# --- SHAP için train'den örneklem ---
rng = np.random.RandomState(42)
sample_size = min(2000, len(X_train))
sample_idx = rng.choice(len(X_train), size=sample_size, replace=False)

X_shap = X_train.iloc[sample_idx]
X_shap_trans = pre.transform(X_shap)

# Sparse -> dense
try:
    X_shap_dense = X_shap_trans.toarray()
except:
    X_shap_dense = np.asarray(X_shap_trans)

# SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap_dense)

# expected value
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[0]

print("SHAP hazır. sample_size:", sample_size)
print("Feature count:", len(feature_names), "Transformed shape:", X_shap_dense.shape)

SHAP hazır. sample_size: 2000
Feature count: 51 Transformed shape: (2000, 51)


In [47]:
plt.figure()
shap.summary_plot(
    shap_values,
    X_shap_dense,
    feature_names=feature_names,
    show=False,
    max_display=15
)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=200, bbox_inches="tight")
plt.close()

print("Kaydedildi: shap_summary.png")

Kaydedildi: shap_summary.png


In [48]:
# bank-full'da numeric kolonlar genelde num__age, num__duration şeklinde gelir
duration_name = "num__duration"
age_name = "num__age"

# Eğer hata olursa, şu satırla doğru isimleri bul:
# [x for x in feature_names if "duration" in x or "age" in x]

plt.figure()
shap.dependence_plot(
    duration_name,
    shap_values,
    X_shap_dense,
    feature_names=feature_names,
    interaction_index=age_name,
    show=False
)
plt.tight_layout()
plt.savefig("shap_dependence_duration_age.png", dpi=200, bbox_inches="tight")
plt.close()

print("Kaydedildi: shap_dependence_duration_age.png")

Kaydedildi: shap_dependence_duration_age.png


<Figure size 640x480 with 0 Axes>

In [49]:
# Test set olasılıkları (yes class)
proba = best_pipeline.predict_proba(X_test)[:, 1]

idx_high = int(np.argmax(proba))                 # en yüksek P(yes)
idx_low  = int(np.argmin(proba))                 # en düşük P(yes)
idx_mid  = int(np.argmin(np.abs(proba - 0.5)))   # 0.5'e en yakın (kararsız)

picked = [idx_high, idx_mid, idx_low]
labels = ["High-Yes (Abone olur)", "Borderline (Kararsız)", "Low-Yes (Abone olmaz)"]

print("Seçilen indeksler:", picked)
print("Olasılıklar:", [float(proba[i]) for i in picked])

Seçilen indeksler: [2399, 7657, 578]
Olasılıklar: [0.9955592155456543, 0.5000541806221008, 3.981684130849317e-06]


In [50]:
# Seçilen 3 müşteri örneğini preprocess et
X_pick = X_test.iloc[picked]
X_pick_trans = pre.transform(X_pick)

try:
    X_pick_dense = X_pick_trans.toarray()
except:
    X_pick_dense = np.asarray(X_pick_trans)

# Bu 3 örnek için shap değerleri
shap_pick = explainer.shap_values(X_pick_dense)

# HTML oluştur (3 force plot tek dosyada)
html_parts = []
html_parts.append("<html><head><meta charset='utf-8'></head><body>")
html_parts.append("<h2>SHAP Force Plots (3 Müşteri Profili)</h2>")
html_parts.append(shap.getjs())

for i in range(3):
    html_parts.append(f"<h3>{labels[i]} — P(yes)={proba[picked[i]]:.3f}</h3>")
    fp = shap.force_plot(
        expected_value,
        shap_pick[i, :],
        X_pick_dense[i, :],
        feature_names=feature_names,
        matplotlib=False
    )
    html_parts.append(fp.html())

html_parts.append("</body></html>")

with open("shap_force_plot.html", "w", encoding="utf-8") as f:
    f.write("\n".join(html_parts))

print("Kaydedildi: shap_force_plot.html")

Kaydedildi: shap_force_plot.html


In [52]:
print(os.listdir())

['.ipynb_checkpoints', 'bank-full.csv', 'best_pipeline.pkl', 'Hafta1_Odev.ipynb', 'model_comparison.csv', 'shap_dependence_duration_age.png', 'shap_force_plot.html', 'shap_summary.png']
